# 01 | Land source data in Bronze
**Purpose:** Read the supplied CSV files from `Files/source/`, preserve their original column values, and record lineage. Run with `batch_id = "01"` first, then `"02"` to demonstrate incremental ingestion. Re-running the same batch does not append it twice.

Attach `lh_retail_medallion` as the **default lakehouse** before running.

In [ ]:
# PARAMETERS — also mark this cell as a parameter cell in Fabric if using pipeline overrides.
batch_id = "01"  # valid values: "01" and "02"


In [ ]:
from pyspark.sql import functions as F

assert batch_id in ("01", "02"), "batch_id must be 01 or 02"
base = "Files/source"
required = ["customers.csv", "products.csv", f"orders_batch_{batch_id}.csv"]
for filename in required:
    path = f"{base}/{filename}"
    assert notebookutils.fs.exists(path), f"Missing source file: {path}"

def read_source(filename):
    return (spark.read.option("header", True).option("inferSchema", False)
            .option("mode", "FAILFAST").csv(f"{base}/{filename}"))

for source, table in [("customers.csv", "bronze_customers"), ("products.csv", "bronze_products")]:
    df = (read_source(source).withColumn("_source_file", F.lit(source))
          .withColumn("_ingested_at", F.current_timestamp()))
    df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(table)
    print(table, df.count())

source_name = f"orders_batch_{batch_id}.csv"
new_rows = (read_source(source_name).withColumn("_source_file", F.lit(source_name))
            .withColumn("_ingested_at", F.current_timestamp()))
if spark.catalog.tableExists("bronze_orders"):
    existing = spark.table("bronze_orders")
    already_loaded = existing.filter(F.col("_source_file") == source_name).limit(1).count() > 0
    if already_loaded:
        print(f"SKIP: {source_name} was already ingested; no duplicate batch added")
    else:
        new_rows.write.format("delta").mode("append").saveAsTable("bronze_orders")
        print(f"APPENDED: {source_name}, rows={new_rows.count()}")
else:
    new_rows.write.format("delta").mode("overwrite").saveAsTable("bronze_orders")
    print(f"CREATED bronze_orders, rows={new_rows.count()}")
print("Bronze order count:", spark.table("bronze_orders").count())
